# Fetch Candidate Resumes

Goal:
- Fetch board candidate profiles and resumes from CoreSignal API
- These profiles will be stored locally under `spi/coresignal/employee_multi_source`

In [ ]:
import json
import os

import requests

from recruit import Candidates

In [ ]:
search: dict = {
    "query": {
        "bool": {
            "filter": [
                {"term": {"is_working": 1}},
                {"term": {"is_deleted": 0}},
                {"term": {"is_parent": 1}},
                {
                    "range": {
                        "total_experience_duration_months": {
                            "gte": 0,
                            "lte": 100,
                        }
                    }
                },
                {
                    "nested": {
                        "path": "languages",
                        "query": {
                            "bool": {
                                "must": [
                                    {
                                        "bool": {
                                            "should": [
                                                {
                                                    "match_phrase": {
                                                        "languages.language": "English"
                                                    }
                                                },
                                                {
                                                    "match_phrase": {
                                                        "languages.language": "Englisch"
                                                    }
                                                },
                                            ],
                                            "minimum_should_match": 1,
                                        }
                                    }
                                ],
                                "must_not": [
                                    {
                                        "terms": {
                                            "languages.proficiency.exact": [
                                                "Elementary proficiency",
                                                "Limited working proficiency",
                                            ]
                                        }
                                    }
                                ],
                            }
                        },
                    }
                },
                {
                    "nested": {
                        "path": "languages",
                        "query": {
                            "bool": {
                                "must": [
                                    {
                                        "bool": {
                                            "should": [
                                                {
                                                    "match_phrase": {
                                                        "languages.language": "German"
                                                    }
                                                },
                                                {
                                                    "match_phrase": {
                                                        "languages.language": "Deutsch"
                                                    }
                                                },
                                            ],
                                            "minimum_should_match": 1,
                                        }
                                    }
                                ],
                                "must_not": [
                                    {
                                        "terms": {
                                            "languages.proficiency.exact": [
                                                "Elementary proficiency",
                                                "Limited working proficiency",
                                            ]
                                        }
                                    }
                                ],
                            }
                        },
                    }
                },
            ],
            "must_not": [
                {"terms": {"location_country": ["India"]}},
                {"terms": {"location_country_iso2": ["IN"]}},
                {"terms": {"location_country_iso3": ["IND"]}},
            ],
            "should": [
                # {"match_phrase": {"location_city": {"query": "Berlin", "boost": 30}}},
                # {"term": {"location_country_iso2": {"value": "DE", "boost": 15}}},
                {
                    "nested": {
                        "path": "education",
                        "score_mode": "max",
                        "query": {
                            "simple_query_string": {
                                "query": '"Software Engineering" | "Artificial Intelligence" | "Mathematics" | "Computer Science" | "Data Science" | "Business Administration"',
                                "fields": ["education.degree"],
                                "default_operator": "or",
                            }
                        },
                        "boost": 10,
                    }
                },
                {
                    "nested": {
                        "path": "experience",
                        "score_mode": "max",
                        "query": {
                            "simple_query_string": {
                                "query": '"Associate" | "Analyst" | "Coordinator" | "Operations" | "Consultant" | "Chief of Staff" | "Project Manager" | "Founder\'s Associate" | "Founder\u2019s Associate"',
                                "fields": ["experience.position_title"],
                                "default_operator": "or",
                            }
                        },
                        "boost": 5,
                    }
                },
            ],
        }
    }
}

In [ ]:
search["sort"] = ["_score"]

In [ ]:
url = "https://api.coresignal.com/cdapi/v2/employee_multi_source/search/es_dsl"
payload = json.dumps(search)

headers = {
    "Content-Type": "application/json",
    "apikey": os.environ["CORESIGNAL_API_KEY"],
}

response = requests.request("POST", url, headers=headers, data=payload)
person_ids: list[int] = json.loads(response.text)

print(len(person_ids))

In [ ]:
if 485761352 in person_ids:
    print("Found 485761352")

In [ ]:
person_ids[:10]

In [ ]:
candidates = Candidates()
candidates.get_persons(person_ids[:100])